# Hi!!!
## And welcome to: <i>"The Science Behind AI Voice Processing Hands-on Workshop"</i>!

## Also important!
1. Create your own virtual environment to run this lab after cloning the repository and opening it on VS Code:  
1.1. On your terminal run:
```cmd
            py -m venv .venv
```
1.2. Activate the environment (syntax depends on OS and type of terminal):
```cmd
            .\\.venv\\Scripts\\activate.bat
```
2. Make sure you run all library requirements installation:
```cmd
            pip install -r requirements.txt
```
3. Generate your own personal Gemini AI API Token: https://aistudio.google.com/api-keys  
    &lt;click at "Get API Key"&gt;  
    &lt;click at "Create API Key"&gt;  
    Keys usually begin with `AIza...`

    **Note:** <u>Keep it to yourself, don't share it or commit it to git!</u>

4. Store your key in a `.env` file at the project root:
```
            # Copy .env.example → .env, then paste your key:
            GOOGLE_API_KEY=AIza...your_key_here...
```
    The cell below will confirm it loaded correctly. Every notebook picks it up automatically — you only need to set it once.


### Run this BEFORE the workshop!

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — One-time setup: put your Gemini key in the project's .env file
#
#   1. Copy  .env.example  →  .env  (project root)
#   2. Edit .env and replace the placeholder:
#        GOOGLE_API_KEY=AIza...your_key_here...
#   3. Re-run this cell — it will confirm the key was found.
#
#   .env is gitignored: your key will never be committed to the repo.
#   Every notebook calls load_dotenv() so you only set the key once.
# ─────────────────────────────────────────────────────────────────────────────
load_dotenv(find_dotenv())   # walks up from cwd until it finds .env
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

if os.environ.get("GOOGLE_API_KEY"):
    print("✅ GOOGLE_API_KEY loaded from .env")
else:
    print("⚠️  GOOGLE_API_KEY not found.")
    print("   Create a .env file in the project root containing:")
    print("   GOOGLE_API_KEY=AIza...your_key_here...")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — Environment health check (every library used across all 5 notebooks)
# ─────────────────────────────────────────────────────────────────────────────
checks = {
    # Core audio + signal processing
    "numpy":               "numpy",
    "librosa":             "librosa",
    "sounddevice":         "sounddevice",
    "soundfile":           "soundfile",
    "scipy":               "scipy",
    "noisereduce":         "noisereduce",
    "tabulate":            "tabulate",
    # Machine learning
    "sklearn":             "scikit-learn",
    "pandas":              "pandas",
    # Deep learning
    "torch":               "torch",
    "torchaudio":          "torchaudio",
    "transformers":        "transformers",
    "audioseal":           "audioseal",
    "pyaudio":             "pyaudio",
    # AI services & TTS
    "google.generativeai": "google-generativeai",
    "edge_tts":            "edge-tts",
    "dotenv":              "python-dotenv",
    # Notebook display
    "matplotlib":          "matplotlib",
    "IPython":             "ipython",
}

print("\n🔍 Checking all dependencies...\n")
missing = []
for module, package in checks.items():
    try:
        __import__(module)
        print(f"  ✅  {package}")
    except ImportError:
        print(f"  ❌  {package}")
        missing.append(package)

if missing:
    print(f"\n⚠️  Missing: {', '.join(missing)}")
    print("   Fix with: pip install -r requirements.txt")
else:
    print("\n✅ All dependencies are present!")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — Pre-download SmolLM2-360M for offline use in Part 4
# ─────────────────────────────────────────────────────────────────────────────
from transformers import pipeline

print("\n📥 Pre-downloading SmolLM2-360M to local cache...")
pipeline("text-generation", model="HuggingFaceTB/SmolLM2-360M-Instruct")
print("🎯 SmolLM2 is cached and ready for offline use in Part 4.")

## 🗂️ Step 4: Create Workshop Folder Structure

Creates the output directories that notebooks 2–5 write into.
These folders are gitignored — this cell ensures they exist on your machine after cloning.

In [ ]:
from pathlib import Path

WORKSHOP_DIR = Path("..").resolve()

print("📁 Creating workshop folder structure...\n")
for name in ["librispeech", "mfcc", "models", "stt-tts"]:
    (WORKSHOP_DIR / name).mkdir(exist_ok=True)
    print(f"  ✅  {name}/")
print("\n✅ All folders ready.")

## ⬇️ Step 5: Pre-Download Models & Dataset

Downloads all heavy assets used in Parts 4 and 5. Run once — everything is cached and won't re-download on subsequent runs.

| Asset | Size | Used in |
|-------|------|---------|
| WavLM-Base-Plus-SV | ~377 MB | Part 5 — Speaker embeddings |
| AudioSeal (generator + detector) | ~150 MB | Part 5 — Neural watermarking |
| LibriSpeech test-clean | ~346 MB | Part 5 — Voice biometrics dataset |

**Total: ~873 MB.** Grab a coffee. ☕

In [ ]:
from transformers import AutoFeatureExtractor, WavLMForXVector
from audioseal import AudioSeal

print("📥 Downloading WavLM-Base-Plus-SV (~377 MB)...")
AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus-sv")
WavLMForXVector.from_pretrained("microsoft/wavlm-base-plus-sv")
print("✅ WavLM-Base-Plus-SV ready.\n")

print("📥 Downloading AudioSeal models (~150 MB)...")
AudioSeal.load_generator("audioseal_wm_16bits")
AudioSeal.load_detector("audioseal_detector_16bits")
print("✅ AudioSeal models ready.")

In [ ]:
import torchaudio
import soundfile as sf

LIBRISPEECH_ROOT         = WORKSHOP_DIR / "librispeech"
AUDIO_DIR                = WORKSHOP_DIR / "audio"
N_SPEAKERS_TO_USE        = 10
N_UTTERANCES_PER_SPEAKER = 3

LIBRISPEECH_ROOT.mkdir(exist_ok=True)
AUDIO_DIR.mkdir(exist_ok=True)

if any(AUDIO_DIR.glob("speaker_*.wav")):
    print("✅ LibriSpeech WAV files already in audio/ — skipping download.")
else:
    print("📥 Downloading LibriSpeech test-clean (~346 MB, first run only)...")
    _ = torchaudio.datasets.LIBRISPEECH(
        root=str(LIBRISPEECH_ROOT),
        url="test-clean",
        download=True,
    )
    ls_root      = LIBRISPEECH_ROOT / "LibriSpeech" / "test-clean"
    all_speakers = sorted(p.name for p in ls_root.iterdir() if p.is_dir())
    selected     = all_speakers[:N_SPEAKERS_TO_USE]
    print(f"\nConverting {len(selected)} speakers × {N_UTTERANCES_PER_SPEAKER} utterances → audio/...\n")
    for spk in selected:
        flac_files = sorted((ls_root / spk).rglob("*.flac"))[:N_UTTERANCES_PER_SPEAKER]
        for i, src in enumerate(flac_files, start=1):
            audio, sr = sf.read(src)
            sf.write(AUDIO_DIR / f"speaker_{spk}_utt_{i}.wav", audio, sr, subtype="PCM_16")
    n = len(list(AUDIO_DIR.glob("speaker_*.wav")))
    print(f"✅ Converted: {n} WAV files saved to audio/")

# Done! Have fun! :)